In [ ]:
# 데이터 탐색 및 준비
import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs, make_classification, make_regression

# 변환(정규화, 표준화)
from sklearn.preprocessing import MinMaxScaler, StandardScaler
# 축소
from sklearn.decomposition import PCA
# 분리(train, test)
from sklearn.model_selection import train_test_split

In [ ]:
# 분류 데이터 생성
# X: 독립(입력)변수
# y: 종속(출력)변수
X, y = make_classification(n_samples=100, n_features=5, random_state=42)
print(X)
print(y)

In [ ]:
X.shape

In [ ]:
df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(X.shape[1])])
df

In [ ]:
df['target'] = y

In [ ]:
# 분류 데이터 생성 -> 결측치 데이터 추가
# Helper function to create DataFrame
def create_classification_data():
    X, y = make_classification(n_samples=100, n_features=5, random_state=42)
    df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(X.shape[1])])
    df['target'] = y
    
    # feature_1에 결측치를 추가 (10% 비율로)
    # replace=False: 중복 없이
    missing_indices = np.random.choice(df.index, size=int(len(df) * 0.1), replace=False)
    df.loc[missing_indices, 'feature_1'] = np.nan
    
    return df

In [ ]:
create_classification_data()

In [ ]:
X, y = make_regression(n_samples=100, n_features=5, noise=0.1, random_state=42)
print(X)
print(y)

In [ ]:
df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(X.shape[1])])
df['target'] = y
df

In [ ]:
# 회귀 데이터 생성 -> 이상치 데이터 추가
def create_regression_data():
    X, y = make_regression(n_samples=100, n_features=5, noise=0.1, random_state=42)
    df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(X.shape[1])])
    df['target'] = y

    # IQR(Interquartile Range): 이상치 포함여부 판단
    for column in df.columns[:-1]:
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1

        # 하한선
        lower_bound = Q1 - 1.5 * IQR
        # 상한선
        upper_bound = Q3 + 1.5 * IQR

        # 이상치를 추가할 인덱스를 랜덤하게 선택
        outlier_indices = np.random.choice(df.index, size=5, replace=False)
        for idx in outlier_indices:
            # at(행, 열)
            df.at[idx, column] = np.random.uniform(upper_bound + 1, upper_bound + 10)

    return df

In [ ]:
create_regression_data()

In [ ]:
# 군집 데이터 생성 함수
def create_blobs_data():
    X, y = make_blobs(n_samples=100, n_features=5, centers=3, random_state=42)
    df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(X.shape[1])])
    df['target'] = y
    
    return df

In [ ]:
create_blobs_data()

In [ ]:
# Generate datasets for each problem type
classification_df = create_classification_data()
regression_df = create_regression_data()
blobs_df = create_blobs_data()

print('=== Classification Generated Datasets ===')
print(classification_df.head())
print('\n=== regression Generated Datasets ===')
print(regression_df.head())
print('\n=== blobs Generated Datasets ===')
print(blobs_df.head())

## 데이터 정제
### 1. 결측치 제거

In [ ]:
# 결측치 확인
classification_df.isna()

In [ ]:
print('[전처리 전] 결측치 수:', classification_df.isna().sum())

In [ ]:
print('[전처리 전] 결측치 수:', classification_df.isna().sum().sum())

In [ ]:
# 결측치를 feagure_1의 평균값으로 변경
classification_df['feature_1'] = classification_df['feature_1'].fillna(classification_df['feature_1'].mean())
print('[전처리 후] 결측치 수:', classification_df.isna().sum().sum())

### 2. 이상치 제거

In [ ]:
print('이상치 처리 전 결측치 수:', regression_df['feature_3'].isna().sum())

In [ ]:
# IQR
Q1 = regression_df['feature_3'].quantile(0.25)
Q3 = regression_df['feature_3'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 이상치를 결측치로 변환
regression_df.loc[(regression_df['feature_3'] < lower_bound) | (regression_df['feature_3'] > upper_bound), 'feature_3'] = None

print('이상치 처리 후 결측치 수:', regression_df['feature_3'].isna().sum())

In [ ]:
regression_df.info()

In [ ]:
regression_df.dropna(subset=['feature_3'], inplace=True)
print('결측치 삭제:', regression_df['feature_3'].isna().sum())

In [ ]:
regression_df.info()

## 데이터 통합

In [ ]:
# 수평 결합
# axis=1: 결합의 축을열(Column) 방향
pd.concat([classification_df, regression_df], axis=1)

In [ ]:
# 병합(조인)
# on='target': 병합 기준 열
pd.merge(classification_df, blobs_df, on='target', how='inner')

## 데이터 탐색(EDA)

In [ ]:
# 그룹별 평균
classification_df.groupby('target')[['feature_1', 'feature_2', 'feature_3']].mean()

## 데이터 변환

In [ ]:
blobs_df

In [ ]:
# 원-핫 인코딩
# prefix: 기본값은 열이름_순번
pd.get_dummies(blobs_df, columns=['target'], prefix='t')

In [ ]:
# feature_6 = feature_1 + feature_2
blobs_df['feature_6'] = blobs_df['feature_1'] + blobs_df['feature_2']

# feature_7 = (feature_3 + feature_4 + feature_5) / 3
blobs_df['feature_7'] = blobs_df[['feature_3', 'feature_4', 'feature_5']].mean(axis=1)

blobs_df

In [ ]:
# 정규화: 0~1 사이값으로 변환
scaler = MinMaxScaler()
regression_df[['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5']] = scaler.fit_transform(regression_df[['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5']])

regression_df

In [ ]:
# 표준화: 평균 0, 표준편차 1, 표준정규분포
scaler = StandardScaler()
classification_df[['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5']] = scaler.fit_transform(classification_df[['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5']])

classification_df

## 데이터 축소

In [ ]:
# PCA(주성분 분석)
pca = PCA(n_components=3)
principal_components = pca.fit_transform(regression_df[['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5']])

principal_df = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2', 'PC3'])

principal_df

In [ ]:
classification_df

In [ ]:
X = classification_df.drop(columns=['target'])
y = classification_df['target']

# 훈련 세트(Train Set)와 테스트 세트(Test Set) 두 개로 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)